# 1. Public APIs related to my project

My project has to do with lung cancer and pollution. I've found datasets to use, with the data I need, but these APIs might provide supplementary information:
- PM2.5 Open Data Portal: https://pm25.lass-net.org/#apis
- AirNow US EPA air quality data and forecasts: https://docs.airnowapi.org/

# 2. Websites for Scraping

American Lung Association's "Most Polluted American Cities" table in its "State of the Air"website.\
Check out the last table, "By Year-Round Particle Pollution"

https://www.lung.org/research/sota/city-rankings/most-polluted-cities

My project looks at air pollution by county, but another approach might be to look at it by city, since I think most people are more familiar with that categorization scheme. So the American Lung Association data might be useful in this regard. The issue is that I'd want to go back to historical air pollution data. The ALA started tracking this in 2004, but it's in a .pdf format and only shows a table for the 25 worst cities. (To do the analysis, I'd need to get all major US cities. I could do more research to find out if it lives on the site somewhere.)

# 3. Using the MET API

Instructions: Use the MET API to assemble a new list of dictionaries containing only items that relate to a specific animal, person, place. This doesn’t have to be comprehensive, as you may end up with thousands of items, so make a list that has at least 5 items in it.

## (Step 1) Use the MET API to get access to all items with Napoleon referenced

In [26]:
import requests #This brings in the functions needed to grab data from the web and deal with responses
from time import sleep #This allows me to fetch all the relevant data without timing out

base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"
search_query = "?q=Napoleon" # I want to look for artworks that feature Napoleon.
date_range = "&dateBegin=1769&dateEnd=2000" #Napoleon was born in 1769 so only look for works after that (although this probably isn't necessary)

url = f'{base_url}{search_path}{search_query}{date_range}'
search_response = requests.get(url)

print(url)   # https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Antelope

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Napoleon&dateBegin=1769&dateEnd=2000


In [27]:
print(met_objects["total"]) # shows us what's in the "total" key (total amount of objects with the search criteria)

862


In [29]:
napoleon_object_ids = (met_objects['objectIDs'])
print(len(napoleon_object_ids)) # This now contains a list of MET Object IDs representing Napoleon-related works in the first batch of data. 

# Out of the 862 Napoleon-related items the Met contains, I am looking at the first 100. But why can't I see all 862 if I started the code with from time import? 

100


In [30]:
object_path = "/public/collection/v1/objects/" # this is the path of the object's endpoint in the MET api

napoleon_items = napoleon_object_ids[0] # the first item in the list of objecct IDs above

url = f"{base_url}{object_path}{napoleon_items}"

response_object = requests.get(url)

print(response_object.ok)

True


In [31]:
print(url)
# Interesting -- I just looked at this webpage and saw that the item is a self-portrait by an artists named Napoleon Sarony. 
# So clearly I need to find a way to refine my search! For the sake of this exercise I'm going to continue...


https://collectionapi.metmuseum.org/public/collection/v1/objects/909965


## (Step 2) Now look for all the details about the first 10 items in the Napoleon list

In [25]:
from time import sleep


In [36]:
object_path = "/public/collection/v1/objects/"

all_napoleon_details = []

for napoleon_id in napoleon_object_ids[:10]: # for THE FIRST 10 ITEMS of the object IDs I pulled from the MET api earlier
    url = f"{base_url}{object_path}{napoleon_id}"
    response_object = requests.get(url) # making the request to the server
    all_napoleon_details.append(response_object.json())
    print(f"grabbed details from object id{napoleon_id} with status code {response_object.status_code}")
    sleep(0.5) # wait half a sectond before moving on to the next step in the for loop

print(len(all_napoleon_details))

grabbed details from object id909965 with status code 200
grabbed details from object id283247 with status code 200
grabbed details from object id269216 with status code 200
grabbed details from object id271534 with status code 200
grabbed details from object id285691 with status code 200
grabbed details from object id285692 with status code 200
grabbed details from object id381079 with status code 200
grabbed details from object id383222 with status code 200
grabbed details from object id660429 with status code 200
grabbed details from object id893371 with status code 200
10


In [52]:
from pprint import pprint # I used Claude to help me do this

pprint(all_napoleon_details[0]) # Get all the details for the first item in the list

{'GalleryNumber': '',
 'accessionNumber': '22.101',
 'accessionYear': '1922',
 'additionalImages': [],
 'artistAlphaSort': 'Sarony, Napoleon',
 'artistBeginDate': '1821',
 'artistDisplayBio': 'American (born Canada), Quebec 1821–1896 New York',
 'artistDisplayName': 'Napoleon Sarony',
 'artistEndDate': '1896',
 'artistGender': '',
 'artistNationality': 'American, born Canada',
 'artistPrefix': '',
 'artistRole': 'Artist',
 'artistSuffix': '',
 'artistULAN_URL': 'http://vocab.getty.edu/page/ulan/500027268',
 'artistWikidata_URL': 'https://www.wikidata.org/wiki/Q965637',
 'city': '',
 'classification': 'Prints',
 'constituents': [{'constituentID': 162378,
                   'constituentULAN_URL': 'http://vocab.getty.edu/page/ulan/500027268',
                   'constituentWikidata_URL': 'https://www.wikidata.org/wiki/Q965637',
                   'gender': '',
                   'name': 'Napoleon Sarony',
                   'role': 'Artist'},
                  {'constituentID': 162378,
  

## (Step 3) Pull out key information about ALL 100 items in the list
Note: I used Claude to help me with this

In [51]:
fields = ["objectID", "artistAlphaSort", "objectName", "medium", "objectDate", "objectURL"] # These are the fields I'm interested in

napoleon_summary = []

for item in all_napoleon_details:
    summary = {}
    for field in fields:
        summary[field] = item.get(field)
    napoleon_summary.append(summary)

print(len(napoleon_summary))
napoleon_summary
          

10


[{'objectID': 909965,
  'artistAlphaSort': 'Sarony, Napoleon',
  'objectName': 'Print',
  'medium': 'Lithograph',
  'objectDate': 'ca. 1873',
  'objectURL': 'https://www.metmuseum.org/art/collection/search/909965'},
 {'objectID': 283247,
  'artistAlphaSort': 'Sarony, Napoleon',
  'objectName': 'Photograph',
  'medium': 'Albumen silver print',
  'objectDate': '1882',
  'objectURL': 'https://www.metmuseum.org/art/collection/search/283247'},
 {'objectID': 269216,
  'artistAlphaSort': 'Sarony, Napoleon',
  'objectName': 'Cabinet card',
  'medium': 'Albumen silver print from glass negative',
  'objectDate': '1880',
  'objectURL': 'https://www.metmuseum.org/art/collection/search/269216'},
 {'objectID': 271534,
  'artistAlphaSort': 'Sarony, Napoleon',
  'objectName': 'Cabinet card',
  'medium': 'Albumen silver print from glass negative',
  'objectDate': 'ca. 1882',
  'objectURL': 'https://www.metmuseum.org/art/collection/search/271534'},
 {'objectID': 285691,
  'artistAlphaSort': 'Sarony, Nap

# 4. Using NYTimes API to research project
Instructions: Use the NYTimes API to search for articles related to your final project research topic within a specific time period. Assemble all of the abstract data into a new list. You will search for specific “terms” in the data that are relevant – for instance "A.I.", counting how many abstracts contain the term. For each abstract in the list, you will determine if there is a match within the string, in which case you will add 1 to the count. Conclude with an f-string that reads "The term {search_term} appears {count} times in the abstracts". (pushed to next week! optional homework for this week.)

## (Step 1) get the relevant abstracts

In [67]:
import requests
from time import sleep

api_key = "bn8h4Yf9hqIduvdG6YKSMMV70ojeW60uhO6X5nmAORdeGBd4"
base_url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

all_articles = []   # an empty list to hold the articles

for page in range(100):   # choosing to go up to 100 pages (I did an earlier version of this code and found that was sufficient)
    params = {
        "q": '"lung cancer" "pollution"',   # what to search for
        "begin_date": "20000101",           # from Jan 1, 2000
        "end_date": "20251231",             # to Dec 31, 2025
        "page": page,                       # which page of results
        "api-key": api_key,
    }
    response = requests.get(base_url, params=params)   # ask the NYT for this page
    print(f"page {page}: status code {response.status_code}")

    docs = response.json()["response"]["docs"]   # the articles on this page
    if not docs:      # if the page is empty, stop
        break
    all_articles.extend(docs)   # add this page's articles to the list
    sleep(12)                   # wait 12 seconds before asking again

print(len(all_articles))

page 0: status code 200
page 1: status code 200
page 2: status code 200
page 3: status code 200
page 4: status code 200
page 5: status code 200
page 6: status code 200
page 7: status code 200
page 8: status code 200
page 9: status code 200
page 10: status code 200
page 11: status code 200
page 12: status code 200
page 13: status code 200
page 14: status code 200
page 15: status code 200
page 16: status code 200
page 17: status code 200
page 18: status code 200
page 19: status code 200
page 20: status code 200
page 21: status code 200
page 22: status code 200
page 23: status code 200
page 24: status code 200
232


## (Step 2) Assemble abstracts into a new list

In [68]:
abstracts = []   # an empty list to hold the abstracts

for article in all_articles:
    abstract = article.get("abstract")   # get this article's abstract
    if abstract:                         # skip it if it's empty (I guess some articles don't contain abstracts)
        abstracts.append(abstract)

print(len(abstracts))

220


## (Step 3 - version #1) Find abstracts that mention just lung cancer and pollution

In [74]:
relevant_abstracts = []   # an empty list for abstracts that mention both

for abstract in abstracts:
    text = abstract.lower()   # make it lowercase so capital letters don't matter
    if "lung cancer" in text and "pollution" in text:   # both must appear
        relevant_abstracts.append(abstract)

print(len(relevant_abstracts))

3


In [72]:
# The above search only returned 3 abstracts so I want to loosen the search terms.


In [78]:
more_relevant_abstracts = []   # an empty list for abstracts that mention both

for abstract in abstracts:
    text = abstract.lower()   # make it lowercase so capital letters don't matter
    if "lung cancer" in text and ("pollution" in text or "smog" in text or "emissions" in text or "air quality" in text or "co2" in text or "particulate" in text):   # both must appear
        more_relevant_abstracts.append(abstract)

print(len(more_relevant_abstracts))

3


In [79]:
# So expanding the search terms doesn't really make a difference. Alright, let's see these 3 articles.

for article in all_articles:
    abstract = article.get("abstract") or ""   # use an empty string if there's no abstract
    text = abstract.lower()

    if "lung cancer" in text and ("pollution" in text or "smog" in text or "emissions" in text or "air quality" in text or "co2" in text or "particulate" in text):
        print(article["headline"]["main"])   # the headline
        print(article["pub_date"][:10])      # the date (first 10 characters, e.g. 2019-04-15)
        print(abstract)
        print(article["web_url"])            # link to the article
        print()                              # blank line between articles

Study Sees 6,000 Deaths From Power Plants
2002-04-18
Study estimates that pollution from more than 80 power plants owned by eight electric utlities will cause nearly 6,000 premature deaths in year 2007; cites particulate matter caused by emissions of sulfur dioxide and nitrogen oxides, which can cause lung cancer and other respiratory ailments; research was conducted for Rockefeller Family Fund by technical consulting firm Abt Associates Inc; map and chart illustrate findings (M)
https://www.nytimes.com/2002/04/18/us/study-sees-6000-deaths-from-power-plants.html

Soot Particles Strongly Tied To Lung Cancer, Study Finds
2002-03-06
Study of 500,000 people in 116 American cities finds prolonged exposure to air tainted with tiny soot particles significantly raises risk of dying of lung cancer or other lung and heart diseases; finds many city residents face long-term risk of fatal lung cancer similar to that of someone living with smoker; Dr George D Thurston, study leader, says findings ad

## (Step 3 - version #2) Find abstracts OR SNIPPETS OR HEADLINES that mention lung cancer and pollution

In [80]:
relevant_articles = []   # an empty list for articles that mention lung cancer and pollution (in title, abstract or snippet)

for article in all_articles:
    headline = article["headline"]["main"] or ""   # use an empty string if a field is missing
    abstract = article.get("abstract") or ""
    snippet = article.get("snippet") or ""

    text = (headline + " " + abstract + " " + snippet).lower()   # join them into one string, lowercase

    if "lung cancer" in text and ("pollution" in text or "smog" in text or "emissions" in text or "air quality" in text or "co2" in text or "particulate" in text):
        relevant_articles.append(article)

print(len(relevant_articles))

6


In [82]:
# Review the results

for article in relevant_articles:
    print(article["headline"]["main"])
    print(article["pub_date"][:10])
    print(article.get("abstract"))
    print(article["web_url"])
    print()

A Poem Praises Smog, and Why Not? It’s From Cancer’s Perspective
2017-01-06
China’s toxic air has drawn attention to a Chinese surgeon’s poem that takes the viewpoint of a hungry lung cancer reveling in the “delicious mist and haze.”
https://www.nytimes.com/2017/01/06/world/asia/china-smog-cancer-poem.html

Lung Cancer Deaths Soar in China’s Steel Country, Report Says
2016-02-19
Researchers say the cause of the fourfold rise since the 1970s in Hebei Province, a steel-making center, is probably air pollution.
https://www.nytimes.com/2016/02/20/world/asia/china-lung-cancer-deaths-hebei.html

An 8-Year-Old Girl’s Lung Cancer Is Blamed on Air Pollution
2013-11-05
According to her doctor, she has been living near heavily trafficked streets and breathing in the smog that often enshrouds cities in Jiangsu. A recent study found that polluted air had shortened life spans in northern China by five years.
https://sinosphere.blogs.nytimes.com/2013/11/05/an-8-year-old-girls-lung-cancer-is-blamed-on